# RiskLens AI: Explainable Fraud & Risk Intelligence Platform

## 1. Project Overview

Fraud detection is an operational risk problem, not only a classification task. A model that looks strong on accuracy can still be weak for the business if it misses rare fraud cases or creates more alerts than analysts can review.

This MVP uses the IEEE-CIS Fraud Detection transaction data to build a practical fraud risk workflow. The project combines risk scoring, time-based validation, threshold optimization, explainability, and alert prioritization so the output can support fraud investigation decisions.

The central business question is: Which transactions should be prioritized for fraud investigation, why were they flagged, and what trade-off happens when the fraud alert threshold changes?

## 2. Import Libraries

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.alert_generator import build_alert_queue
from src.data_utils import load_transaction_data
from src.evaluation import calculate_metrics, plot_confusion_matrix, plot_precision_recall_curve, plot_roc_curve
from src.explainability import build_reason_lookup, calculate_shap_values, clean_feature_name, get_top_shap_reasons
from src.feature_engineering import (
    TARGET_COLUMN,
    create_features,
    get_model_feature_columns,
    identify_categorical_features,
    identify_numeric_features,
    get_recommended_raw_columns,
    select_features,
)
from src.model_utils import build_preprocessor, save_artifact, train_model
from src.threshold_optimizer import select_best_threshold, simulate_thresholds

plt.style.use("default")
pd.set_option("display.max_columns", 120)

## 3. Load Data

The MVP uses `train_transaction.csv`. The optional `sample_size` parameter keeps the notebook practical on a normal laptop.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "train_transaction.csv"
sample_size = 100000

raw_columns = get_recommended_raw_columns(include_target=True)
data = load_transaction_data(DATA_PATH, sample_size=sample_size, usecols=raw_columns)

print(f"Shape: {data.shape}")
print(f"Fraud ratio: {data[TARGET_COLUMN].mean():.4%}")
display(data.head())
display(pd.DataFrame({"column": data.columns}))
display(data[TARGET_COLUMN].value_counts(normalize=True).rename("share"))
data.info()

## 4. Exploratory Data Analysis

In [ ]:
overall_fraud_rate = data[TARGET_COLUMN].mean()
print(f"Overall fraud rate: {overall_fraud_rate:.4%}")

product_summary = (
    data.groupby("ProductCD")
    .agg(transactions=(TARGET_COLUMN, "size"), fraud_count=(TARGET_COLUMN, "sum"), fraud_rate=(TARGET_COLUMN, "mean"))
    .reset_index()
    .sort_values("fraud_rate", ascending=False)
)
display(product_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(product_summary["ProductCD"], product_summary["fraud_count"])
axes[0].set_title("Fraud Count by ProductCD")
axes[0].set_xlabel("ProductCD")
axes[0].set_ylabel("Fraud Count")

axes[1].bar(product_summary["ProductCD"], product_summary["fraud_rate"])
axes[1].set_title("Fraud Rate by ProductCD")
axes[1].set_xlabel("ProductCD")
axes[1].set_ylabel("Fraud Rate")
plt.tight_layout()
plt.show()

In [ ]:
amount_cap = data["TransactionAmt"].quantile(0.99)
amount_plot_data = data.loc[data["TransactionAmt"] <= amount_cap, ["TransactionAmt", TARGET_COLUMN]].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(amount_plot_data["TransactionAmt"], bins=80)
axes[0].set_title("Transaction Amount Distribution")
axes[0].set_xlabel("TransactionAmt")
axes[0].set_ylabel("Transactions")

amount_plot_data.boxplot(column="TransactionAmt", by=TARGET_COLUMN, ax=axes[1])
axes[1].set_title("TransactionAmt by Fraud Label")
axes[1].set_xlabel("isFraud")
axes[1].set_ylabel("TransactionAmt")
fig.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
missing_summary = (
    data.isna().mean().sort_values(ascending=False).head(20).rename("missing_share").reset_index().rename(columns={"index": "column"})
)
display(missing_summary)

time_pattern = data.assign(transaction_day=data["TransactionDT"] // (3600 * 24)).groupby("transaction_day").agg(
    transactions=(TARGET_COLUMN, "size"), fraud_rate=(TARGET_COLUMN, "mean")
).reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_pattern["transaction_day"], time_pattern["fraud_rate"])
ax.set_title("Fraud Rate Over Transaction Days")
ax.set_xlabel("Transaction Day")
ax.set_ylabel("Fraud Rate")
ax.grid(alpha=0.3)
plt.show()

## 5. Feature Selection

The MVP uses a practical subset of transaction columns rather than all available V columns. This keeps the workflow easier to understand and faster to run.

In [ ]:
selected_data = select_features(data)
print(f"Selected shape: {selected_data.shape}")
display(pd.DataFrame({"selected_column": selected_data.columns}))

## 6. Feature Engineering

The engineered features support time patterns, amount risk, missingness signals, and email consistency.

In [ ]:
model_data, amount_high_threshold = create_features(selected_data)
categorical_features = identify_categorical_features(model_data)
numeric_features = identify_numeric_features(model_data)
feature_columns = get_model_feature_columns(model_data)

print(f"High amount threshold: {amount_high_threshold:,.2f}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
display(model_data.head())

## 7. Train Validation Split

Fraud models should be validated in a way that resembles future deployment. A time-based split trains on earlier transactions and validates on later transactions, which is more realistic than a random split for fraud behavior.

In [ ]:
model_data = model_data.sort_values("TransactionDT").reset_index(drop=True)
split_index = int(len(model_data) * 0.80)

train_df = model_data.iloc[:split_index].copy()
valid_df = model_data.iloc[split_index:].copy()

X_train = train_df[feature_columns]
y_train = train_df[TARGET_COLUMN]
X_valid = valid_df[feature_columns]
y_valid = valid_df[TARGET_COLUMN]

print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {valid_df.shape}")
print(f"Train fraud rate: {y_train.mean():.4%}")
print(f"Validation fraud rate: {y_valid.mean():.4%}")

## 8. Model Development

Two models are trained: a baseline logistic regression model and a main LightGBM model. The main model uses class imbalance handling through `scale_pos_weight`.

In [ ]:
baseline_preprocessor = build_preprocessor(numeric_features, categorical_features, scale_numeric=True)
X_train_baseline = baseline_preprocessor.fit_transform(X_train)
X_valid_baseline = baseline_preprocessor.transform(X_valid)

baseline_model = train_model(X_train_baseline, y_train, model_type="logistic_regression")
baseline_scores = baseline_model.predict_proba(X_valid_baseline)[:, 1]
baseline_metrics = calculate_metrics(y_valid, baseline_scores, threshold=0.50)
display(pd.Series(baseline_metrics, name="baseline_logistic_regression"))

In [ ]:
preprocessor = build_preprocessor(numeric_features, categorical_features, scale_numeric=False)
X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

model = train_model(X_train_processed, y_train, model_type="lightgbm")
valid_scores = model.predict_proba(X_valid_processed)[:, 1]
default_metrics = calculate_metrics(y_valid, valid_scores, threshold=0.50)
display(pd.Series(default_metrics, name="lightgbm_default_threshold"))

## 9. Model Evaluation

Fraud data is highly imbalanced, so accuracy is not the main metric. ROC-AUC and PR-AUC show ranking quality, while precision, recall, F1-score, and the confusion matrix show operational behavior at a chosen threshold.

In [ ]:
display(pd.Series(default_metrics))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
plot_confusion_matrix(y_valid, valid_scores, threshold=0.50, ax=axes[0])
plot_roc_curve(y_valid, valid_scores, ax=axes[1])
plot_precision_recall_curve(y_valid, valid_scores, ax=axes[2])
plt.tight_layout()
plt.show()

## 10. Threshold Optimization

The threshold simulator estimates the business effect of changing the alert threshold. Fraud loss is represented by `TransactionAmt`, investigation cost is set to 5 USD per flagged transaction, and net benefit is captured fraud amount minus investigation cost.

In [ ]:
threshold_results = simulate_thresholds(
    y_true=y_valid,
    y_scores=valid_scores,
    transaction_amounts=valid_df["TransactionAmt"],
    investigation_cost_per_alert=5.0,
)
recommended_threshold = select_best_threshold(threshold_results)
recommended_row = threshold_results.loc[threshold_results["threshold"] == recommended_threshold].iloc[0]

print(f"Recommended threshold: {recommended_threshold:.2f}")
display(threshold_results)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].plot(threshold_results["threshold"], threshold_results["recall"], marker="o")
axes[0].set_title("Threshold vs Recall")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Recall")

axes[1].plot(threshold_results["threshold"], threshold_results["flagged_transactions"], marker="o")
axes[1].set_title("Threshold vs Flagged Transactions")
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Flagged Transactions")

axes[2].plot(threshold_results["threshold"], threshold_results["estimated_net_benefit"], marker="o")
axes[2].set_title("Threshold vs Estimated Net Benefit")
axes[2].set_xlabel("Threshold")
axes[2].set_ylabel("Estimated Net Benefit")
plt.tight_layout()
plt.show()

## 11. Explainability with SHAP

SHAP is used on a small high-risk validation sample to keep runtime practical. Global importance summarizes broad model drivers, and local explanations provide top risk drivers for selected transactions.

In [ ]:
explanation_sample_size = min(1000, len(X_valid))
score_series = pd.Series(valid_scores, index=X_valid.index)
explain_indices = score_series.sort_values(ascending=False).head(explanation_sample_size).index
X_explain = X_valid.loc[explain_indices]

shap_values, transformed_feature_names = calculate_shap_values(model, preprocessor, X_explain)

importance = pd.DataFrame(
    {
        "feature": [clean_feature_name(name, categorical_features) for name in transformed_feature_names],
        "mean_abs_shap": np.abs(shap_values).mean(axis=0),
    }
)
global_importance = importance.groupby("feature", as_index=False)["mean_abs_shap"].sum().sort_values("mean_abs_shap", ascending=False)
display(global_importance.head(20))

fig, ax = plt.subplots(figsize=(8, 6))
top_global = global_importance.head(15).sort_values("mean_abs_shap")
ax.barh(top_global["feature"], top_global["mean_abs_shap"])
ax.set_title("Global SHAP Feature Importance")
ax.set_xlabel("Mean Absolute SHAP Value")
plt.tight_layout()
plt.show()

In [ ]:
reason_lookup = build_reason_lookup(
    valid_df.loc[explain_indices, "TransactionID"],
    shap_values,
    transformed_feature_names,
    top_n=3,
    categorical_features=categorical_features,
)
default_reasons = global_importance["feature"].head(3).tolist()

for index in explain_indices[:5]:
    transaction_id = int(valid_df.loc[index, "TransactionID"])
    risk_score = score_series.loc[index]
    top_reasons = reason_lookup[transaction_id]
    print(f"TransactionID: {transaction_id}")
    print(f"Risk Score: {risk_score:.2f}")
    print("Top Risk Drivers:")
    for rank, reason in enumerate(top_reasons, start=1):
        print(f"{rank}. {reason}")
    print("")

## 12. Fraud Alert Queue

The alert queue converts model scores into a review list for analysts. High, medium, and low risk levels determine recommended action.

In [ ]:
alert_queue = build_alert_queue(
    validation_data=valid_df,
    risk_scores=valid_scores,
    recommended_threshold=recommended_threshold,
    reason_lookup=reason_lookup,
    default_reasons=default_reasons,
)
display(alert_queue.head(20))

## 13. Save Outputs

In [ ]:
models_dir = PROJECT_ROOT / "models"
outputs_dir = PROJECT_ROOT / "outputs"
models_dir.mkdir(exist_ok=True)
outputs_dir.mkdir(exist_ok=True)

recommended_metrics = calculate_metrics(y_valid, valid_scores, threshold=recommended_threshold)
metrics_summary = {
    **recommended_metrics,
    "recommended_threshold": recommended_threshold,
    "total_validation_transactions": int(len(valid_df)),
    "validation_fraud_rate": float(y_valid.mean()),
    "recommended_alerts": int(recommended_row["flagged_transactions"]),
    "estimated_fraud_amount_captured": float(recommended_row["estimated_fraud_amount_captured"]),
    "estimated_net_benefit": float(recommended_row["estimated_net_benefit"]),
    "baseline_roc_auc": baseline_metrics["roc_auc"],
    "baseline_pr_auc": baseline_metrics["pr_auc"],
}

save_artifact(model, models_dir / "fraud_model.pkl")
save_artifact(preprocessor, models_dir / "preprocessor.pkl")
save_artifact(feature_columns, models_dir / "feature_columns.pkl")

threshold_results.to_csv(outputs_dir / "threshold_results.csv", index=False)
alert_queue.to_csv(outputs_dir / "fraud_alert_queue.csv", index=False)
with (outputs_dir / "metrics_summary.json").open("w", encoding="utf-8") as file:
    json.dump(metrics_summary, file, indent=2)

display(pd.Series(metrics_summary))

## 14. Key Findings

RiskLens AI produces transaction-level fraud risk scores and turns them into an analyst-ready alert queue. The LightGBM model ranks transactions by risk, while SHAP explanations highlight the features that drive high-risk cases.

Threshold selection matters because fraud teams operate with limited review capacity. A lower threshold can capture more fraud but creates more false positives and investigation cost. A higher threshold reduces workload but can miss more fraud loss.

The alert queue helps analysts prioritize work by combining risk score, risk level, top reasons, recommended action, and a template-based analyst note.

Limitations of this MVP include simplified cost assumptions, use of `train_transaction.csv` only, no production inference service, no drift monitoring, and no retraining workflow. Planned improvements include identity data integration, Evidently AI monitoring, FastAPI serving, Docker packaging, Gemini-powered analyst summaries, and scheduled retraining.